## Supplementary Table S4: TMEM106B peptide-level pQTL of the coding variant T185S

One row per TMEM106B tryptic peptide quantified by TMT proteomics (the 6 peptides shown in Figure 4b). Association with T185S (rs3173615, chr7:12229791:C:G), with the data preparation and model of Supplementary Figure 4c (`tmem106b/analysis/R/figureS4c.R`, `Supplementary_Figure4_plots.ipynb`): one TMT record per donor, peptide abundance centered and divided by the within-batch SD (SD units), `y ~ genotype + Batch + study + age10 + sex`, HC3-robust joint genotype test. Peptides marked with * are the N-terminal peptides used for the N2 score.
Because the model, units and donor set differ from Figure 4b (raw abundance, no covariates, 664 donors), the effect sizes and P values here are not numerically identical to the values drawn in Figure 4b; which peptides show an effect is the same.


In [ ]:
suppressPackageStartupMessages({
  library(SummarizedExperiment)
  library(tidyverse)
  library(sandwich)
  library(car)
})
# data preparation as in figureS4c.R (Supplementary Figure 4c): one TMT record per donor (round r01 > r03 > r02, lowest batch first)
tmt <- readRDS("tmem106b/tmt/tmt_tmem106b.Rds")
d <- as.data.frame(colData(tmt))
ix <- order(match(d$Round, c("r01", "r03", "r02")), d$Batch, seq_len(nrow(d)))
ix <- ix[!duplicated(d$ProjID[ix])]
donors <- d[ix, ] %>%
  mutate(Batch = factor(Batch), study = factor(study), sex = factor(msex), age10 = (age_death - 90) / 10,
         genotype = factor(rs3173615, levels = c("C/C", "C/G", "G/G")), dosage = as.numeric(genotype) - 1)
abund <- assay(tmt, "peptides")[, ix]
# TMEM106B protein sequence (UniProt Q9NUM4, 274 aa), used to locate each peptide
seq_q9num4 <- "MGKSLSHLPLHSSKEDAYDGVTSENMRNGLVNSEVHNEDGRNGDVSQFPYVEFTGRDSVTCPTCQGTGRIPRGQENQLVALIPYSDQRLRPRRTKLYVMASVFVCLLLSGLAVFFLFPRSIDVKYIGVKSAYVSYDVQKRTIYLNITNTLNITNNNYYSVEVENITAQVQFSKTVIGKARLNNITIIGPLDMKQIDYTVPTVIAEEMSYMYDFCTLISIKVHNIVLMMQVTVTTTYFGHSEQISQERYQYVDCGRNTTYQLGQSEYLNVLQPQQ"
# peptide IDs as used in Figure 4 (N_1-N_4 = N-terminal peptides in residue order, C_core = SAYVSYDVQK, C_boundary = YQYVDCGR)
id_map <- c(EDAYDGVTSENMR = "N_1", NGDVSQFPYVEFTGR = "N_2", DSVTCPTCQGTGR = "N_3", GQENQLVALIPYSDQR = "N_4", SAYVSYDVQK = "C_core", YQYVDCGR = "C_boundary")
peptides <- c("EDAYDGVTSENMR", "NGDVSQFPYVEFTGR", "DSVTCPTCQGTGR", "GQENQLVALIPYSDQR", "SAYVSYDVQK", "YQYVDCGR")
n2_peptides <- c("EDAYDGVTSENMR", "GQENQLVALIPYSDQR")   # the two N-terminal peptides with the lowest missing rate, used for the N2 score

fit_peptide <- function(p) {
  y <- abund[p, ]
  # z score: centered and divided by the within-batch SD, as in figureS4c.R
  z <- (y - mean(y, na.rm = TRUE)) / sigma(lm(y ~ Batch, data = donors))
  v <- donors %>% mutate(z = z) %>% drop_na(z, genotype, Batch, study, age10, sex)
  fit <- lm(z ~ genotype + Batch + study + age10 + sex, data = v)
  # HC3-robust joint test of C/G and G/G versus C/C (2 df), equal to the test in figureS4c.R
  joint <- linearHypothesis(fit, c("genotypeC/G = 0", "genotypeG/G = 0"), vcov. = vcovHC(fit, type = "HC3"), test = "F")
  tibble(peptide = p, n = nrow(v), p = joint[2, "Pr(>F)"], cg_cc = coef(fit)[["genotypeC/G"]], gg_cc = coef(fit)[["genotypeG/G"]],
         per_g = coef(lm(z ~ dosage + Batch + study + age10 + sex, data = v))[["dosage"]],
         missing_pct = 100 * mean(is.na(y[!is.na(donors$genotype)])))   # missing rate among the unique TMT donors with a T185S genotype (same donors as the test)
}
fits <- map_dfr(peptides, fit_peptide)
n_genotyped <- sum(!is.na(donors$genotype))   # unique TMT donors with a T185S genotype

table_s3 <- fits %>%
  mutate(start = str_locate(seq_q9num4, peptide)[, 1], end = start + nchar(peptide) - 1) %>%
  arrange(start) %>%
  transmute(ID = paste0(id_map[peptide], ifelse(peptide %in% n2_peptides, "*", "")),   # peptide ID in residue order, * marks the N_mean peptides
            `Peptide sequence` = peptide,
            `Start (aa)` = start, `End (aa)` = end, `Observations (n)` = n, `Missing values (%)` = round(missing_pct, 1),
            `pQTL P value (T185S, rs3173615)` = signif(p, 3),
            `Coefficient (per G allele)` = round(per_g, 3))

# footnote row (the Excel builder moves rows starting with "Note:" below the table)
note <- "Note: * N-terminal peptides used for the N_mean score. Positions refer to UniProt Q9NUM4."
table_s3 %>% bind_rows(tibble(ID = note)) %>% write_csv("TableS4.csv", na = "")
table_s3


In [ ]:
# checks: every peptide located once in the protein, and the numbers equal the Supplementary Figure 4c results (Core = SAYVSYDVQK, Boundary = YQYVDCGR)
tbl <- read_csv("TableS4.csv", show_col_types = FALSE)
data_rows <- tbl %>% filter(!str_detect(ID, "^Note:"))
s4c <- readRDS("../3_supplementary_figures/Supplementary_Figure4/data/Figure_S4c.rds")$results
same_as_s4c <- function(pep, endpoint) with(fits[fits$peptide == pep, ], isTRUE(all.equal(c(n, p, cg_cc, gg_cc), unlist(s4c[endpoint, c("n", "p_genotype_HC3", "CG_minus_CC", "GG_minus_CC")]), check.attributes = FALSE)))
stopifnot(nrow(data_rows) == 6, !anyNA(data_rows$`Start (aa)`), all(str_count(seq_q9num4, data_rows$`Peptide sequence`) == 1),
          sum(str_detect(data_rows$ID, "[*]$")) == 2, identical(data_rows$ID, c("N_1*", "N_2", "N_3", "N_4*", "C_core", "C_boundary")), same_as_s4c("SAYVSYDVQK", "Core"), same_as_s4c("YQYVDCGR", "Boundary"))
# the effects plotted in Figure 4b (Figure4_plots_preparation.ipynb) come from the same model: per-G-allele coefficient and joint P must agree
f4b <- readRDS("../2_main_figures/Figure4/data/Figure_4b_pqtl_effects.rds")
stopifnot(isTRUE(all.equal(f4b$beta, fits$per_g[match(f4b$peptide, fits$peptide)])), isTRUE(all.equal(f4b$p_joint, fits$p[match(f4b$peptide, fits$peptide)])))
cat("Table S4: 6 peptides,", n_genotyped, "unique TMT donors with genotype; matches Supplementary Figure 4c; checks passed\n")
